This noteboook is used to import the dataset with ALL of the feature selection list to make one overlapping list for one MASTER list (overlapping across all folds within one feature selection method AND between every single feature selection method)

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.svm import SVC, SVR
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from skrebate import ReliefF  # pip install skrebate — real ReliefF
from itertools import combinations
import csv

#Import the appropraite datasets

double_rbf = pd.read_csv("../results/top_hundred_results/results_Double-RBF.csv")['Features']
ensemble = pd.read_csv("../results/top_hundred_results/results_Ensemble.csv")['Features']
lasso = pd.read_csv("../results/top_hundred_results/results_Lasso.csv")['Features']
random_forest = pd.read_csv("../results/top_hundred_results/results_Random Forest.csv")['Features']
relief_f = pd.read_csv("../results/top_hundred_results/results_ReliefF.csv")['Features']
sig_feature = pd.read_csv("../results/top_hundred_results/results_SigFeature.csv")['Features']
svm_rbf = pd.read_csv("../results/top_hundred_results/results_SVM-RBF.csv")['Features']
svm_rfe = pd.read_csv("../results/top_hundred_results/results_SVM-RFE.csv")['Features']

dict_of_methods = {"double_rbf": double_rbf, "ensemble":ensemble, "lasso":lasso, "random_forest":random_forest, "relief_f":relief_f, "sig_feature":sig_feature, "svm_rbf":svm_rbf, "svm_rfe":svm_rfe}
#list_of_methods = [double_rbf, ensemble, lasso, random_forest, relief_f, sig_feature, svm_rbf, svm_rfe]

In [2]:
#Getting the intersection for each method
list_of_rows = []
final_method_results_dict = {}
#Go through every method
for method, method_list in dict_of_methods.items():
  
  #Get all the rows and put them into one list
  for row in method_list:
    list_of_rows.append(row.split(";"))

  #Getting one list of overlap and appending to the final list
  current_overlap = set(list_of_rows[0]).intersection(*list_of_rows[1:])
  print("Current list length for: ", method ,len(current_overlap))
  
  final_method_results_dict[method] = current_overlap

  #reset the list_of_rows
  list_of_rows = []

print("final_method_list length:", len(final_method_results_dict))


#Getting the list of all of the unique genes from across all lists
all_genes = []
for gene_list in list(final_method_results_dict.values()):
  for gene in gene_list:
    all_genes.append(gene)

print(len(all_genes))
gene_set = set(all_genes)

print(len(gene_set))


Current list length for:  double_rbf 45
Current list length for:  ensemble 31
Current list length for:  lasso 5
Current list length for:  random_forest 20
Current list length for:  relief_f 31
Current list length for:  sig_feature 16
Current list length for:  svm_rbf 43
Current list length for:  svm_rfe 8
final_method_list length: 8
199
91


In [35]:
'''Extracting log2fold'''
import pandas as pd

df_gene_map = pd.read_csv('../data/GSE201753/gene_map.csv')

# This creates a dictionary where the key is the symbol and the value is the query ID
gene_map = df_gene_map.set_index('symbol')['query'].to_dict()

gene_id_dict = {}

# Normalize gene_set for safe lookup
gene_set_lookup = {str(g).strip().upper() for g in gene_set}

for gene, gene_id in gene_map.items():
  # Clean the CSV gene symbol the same way
  clean_gene = str(gene).strip().upper()
  
  if clean_gene in gene_set_lookup:
    # Keep the original gene name as the key, or clean_gene if you prefer
    gene_id_dict[gene_id] = gene

print(gene_id_dict)

{9651: 'PLCH2', 55160: 'ARHGEF10L', 1912: 'PHC2', 64744: 'SMAP2', 59269: 'HIVEP3', 256435: 'ST6GALNAC3', 57413: 'TMIGD3', 729867: 'PEA15-AS1', 29097: 'CNIH4', 101927532: 'LINC01736', 4811: 'NID1', 79623: 'GALNT14', 7850: 'IL1R2', 150864: 'FAM117B', 105377046: 'LOC105377046', 55349: 'CHDH', 5627: 'PROS1', 57493: 'HEG1', 56648: 'EIF5A2', 5196: 'PF4', 5473: 'PPBP', 374: 'AREG', 1363: 'CPE', 105377540: 'SAP30-DT', 8819: 'SAP30', 3248: 'HPGD', 9509: 'ADAMTS2', 105377759: 'LOC105377759', 2289: 'FKBP5', 340205: 'TREML1', 129685: 'TAF8', 2852: 'GPER1', 105375259: 'LOC105375259', 252969: 'NEIL2', 63898: 'SH2D4A', 1191: 'CLU', 2395: 'FXN', 687: 'KLF9', 101927086: 'KLF9-DT', 114991: 'ZNF618', 79746: 'ECHDC3', 4360: 'MRC1', 81619: 'TSPAN14', 5728: 'PTEN', 143282: 'FGFBP3', 4791: 'NFKB2', 22876: 'INPP5F', 143458: 'LDLRAD3', 55693: 'KDM4D', 7704: 'ZBTB16', 4837: 'NNMT', 5818: 'NECTIN1', 9332: 'CD163', 56938: 'BMAL2', 196410: 'TMT1B', 3679: 'ITGA7', 57522: 'SRGAP1', 55508: 'SLC35E3', 2322: 'FLT3', 89

In [41]:
'''Log2 fold continued'''

df_deseq2_results = pd.read_csv('../data/GSE201753/deseq2_results.csv')
df_deseq2_results = df_deseq2_results.drop(columns=['baseMean', 'lfcSE', 'stat', 'pvalue', 'padj'])
df_deseq2_results.rename(columns={'Unnamed: 0': 'query'}, inplace=True)
print(df_deseq2_results.head())

#Use the previous gene_id_dict to map only the 91 genes

#First convert to dictionary
deseq2_results_map = df_deseq2_results.set_index('query')['log2FoldChange'].to_dict()
print(deseq2_results_map)
# Now that we have two maps, we match them and make a csv
with open('../results/log2FoldChange_91_genes.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)
  
  writer.writerow(['query', 'gene_name', 'log2FoldChange'])

  for query, log2value in deseq2_results_map.items():
    # 'in gene_id_dict' automatically checks keys; no need for list() or .keys()
    if query in gene_id_dict:
      # Use a single flat list (no double brackets)
      row = [query, gene_id_dict[query], log2value]
      writer.writerow(row)
      

       query  log2FoldChange
0     729737       -0.711643
1  100132287       -0.560997
2  100133331       -0.480835
3     643837       -0.832648
4     284593       -0.888853
{729737: -0.711642784927334, 100132287: -0.560997280403263, 100133331: -0.480835077313939, 643837: -0.832648316806851, 284593: -0.888852876296893, 339451: -0.604164197955024, 84069: -0.941406711611916, 57801: -2.08309085257577, 375790: -0.800021782535391, 100288175: -2.51277323500115, 112267874: -2.74216867669451, 105378948: -2.37279275626262, 254173: -3.03241396684558, 8784: -2.99881784065134, 7293: -0.938627566362555, 51150: -0.460285360087523, 388581: -0.688977151940408, 118424: -0.369983575596865, 54998: 0.321883439985353, 2782: -0.183542788990389, 9651: -0.817969574659938, 105378604: -2.31781382576765, 57212: 0.371846738467709, 387509: -1.4226737407176, 11332: -0.593801536166328, 8718: -1.41115482746791, 148479: -0.927377821132007, 90326: 0.422839915641755, 55735: 0.41004127768023, 8863: 0.649446143163688, 100

In [3]:
#Getting the master intersection 
# Convert the dictionary straight to a DataFrame
# Open a CSV file for writing
with open("GCA_master_gene_list.csv", "w", newline="", encoding="utf-8") as f:
  writer = csv.writer(f)

  # Loop through the dictionary and write each key + list values
  for key, value_list in final_method_results_dict.items():
    writer.writerow([key, *value_list])
  
  writer.writerow(gene_set)